In [4]:
"""
Quick status check: which model/task/variant runs completed all lags & folds?
Only reads CSV headers + lag column — no heavy metric loading.
"""
import pandas as pd
from pathlib import Path
import re
import yaml

RESULTS_ROOT = Path("/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/results/foundation_models")

# Expected parameters (from job_script_in_interactive.sh overrides)
EXPECTED_LAGS = [-250, 0, 250, 500]  # min_lag=-250, max_lag=750, step=250
EXPECTED_FOLDS = [1, 5]

MODELS = ["brainbert", "diver", "popt"]
TASKS = [
    "gpt_surprise", "content_noncontent", "gpt_surprise_multiclass",
    "iu_boundary", "llm_embedding_pretraining",
    "pos", "sentence_onset", "whisper_embedding"
]
#"llm_decoding" "volume_level",  "word_embedding", 
VARIANTS = ["supersubject"]

rows = []
for model in MODELS:
    for task in TASKS:
        for variant in VARIANTS:
            base = RESULTS_ROOT / model / task / variant
            if not base.exists():
                rows.append(dict(model=model, task=task, variant=variant,
                                 status="NO_DIR", lags_done=0, folds_done=0,
                                 missing_lags=EXPECTED_LAGS, latest_run=None))
                continue

            run_dirs = sorted(base.iterdir())
            # Filter: only runs after 1:30 AM EDT Apr 7 2026 (= 10:30 PM PDT Apr 6)
            run_dirs = [d for d in run_dirs if re.search(r'\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}$', d.name)
                        and re.search(r'\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}$', d.name).group() >= "2026-04-07-01-00-00"]
            if not run_dirs:
                rows.append(dict(model=model, task=task, variant=variant,
                                 status="EMPTY_DIR", lags_done=0, folds_done=0,
                                 missing_lags=EXPECTED_LAGS, latest_run=None))
                continue

            # Aggregate lags & folds across ALL run dirs (not just latest)
            all_lags = set()
            all_fold_nums = set()
            latest_name = run_dirs[-1].name
            has_csv = False
            for rd in run_dirs:
                # Skip runs where config.yml epochs != 100
                cfg_path = rd / "config.yml"
                if cfg_path.exists():
                    with open(cfg_path) as f:
                        cfg = yaml.safe_load(f)
                    if cfg.get("training_params", {}).get("epochs") != 100: #! filter out debug runs.p
                        continue
                csv_path = rd / "lag_performance.csv"
                if not csv_path.exists():
                    continue
                has_csv = True
                df = pd.read_csv(csv_path, usecols=lambda c: c == "lags" or re.match(r'test_\w+_fold_\d+', c))
                all_lags.update(df["lags"].tolist())
                fold_cols = [c for c in df.columns if re.match(r'test_\w+_fold_\d+', c)]
                all_fold_nums.update(int(re.search(r'fold_(\d+)', c).group(1)) for c in fold_cols)

            if not has_csv:
                rows.append(dict(model=model, task=task, variant=variant,
                                 status="NO_CSV", lags_done=0, folds_done=0,
                                 missing_lags=EXPECTED_LAGS, latest_run=latest_name))
                continue

            done_lags = sorted(all_lags)
            missing_lags = [l for l in EXPECTED_LAGS if l not in done_lags]
            folds_present = sorted(all_fold_nums)
            missing_folds = [f for f in EXPECTED_FOLDS if f not in folds_present]

            if not missing_lags and not missing_folds:
                status = "DONE"
            elif len(done_lags) == 0:
                status = "FAILED"
            else:
                status = f"PARTIAL ({len(done_lags)}/{len(EXPECTED_LAGS)} lags, {len(folds_present)}/{len(EXPECTED_FOLDS)} folds)"

            rows.append(dict(model=model, task=task, variant=variant,
                             status=status, lags_done=len(done_lags),
                             folds_done=len(folds_present),
                             missing_lags=missing_lags, latest_run=latest_name))

status_df = pd.DataFrame(rows)

# Summary
print("=== Run Status Summary ===")
print(status_df["status"].value_counts().to_string())
print()

complete = status_df[status_df["status"] == "DONE"]
if not complete.empty:
    print(f"{len(complete)} complete runs:")
    display(complete[["model", "task", "variant", "status", "lags_done", "folds_done", "latest_run"]])
else:
    print("No complete runs yet.")
print()
complete

incomplete = status_df[status_df["status"] != "DONE"]
if incomplete.empty:
    print("All runs complete!")
else:
    print(f"{len(incomplete)} incomplete runs:")
    display(incomplete[["model", "task", "variant", "status", "lags_done", "folds_done", "missing_lags"]])

status_df

=== Run Status Summary ===
status
DONE    24

24 complete runs:


,model,task,variant,status,lags_done,folds_done,latest_run
0,brainbert,gpt_surprise,supersubject,DONE,4,2,brainbert_supersubject_gpt_surprise_2026-04-07...
1,brainbert,content_noncontent,supersubject,DONE,4,2,brainbert_supersubject_content_noncontent_2026...
2,brainbert,gpt_surprise_multiclass,supersubject,DONE,4,2,brainbert_supersubject_gpt_surprise_multiclass...
3,brainbert,iu_boundary,supersubject,DONE,4,2,brainbert_supersubject_iu_boundary_2026-04-07-...
4,brainbert,llm_embedding_pretraining,supersubject,DONE,4,2,brainbert_supersubject_llm_embedding_pretraini...
5,brainbert,pos,supersubject,DONE,4,2,brainbert_supersubject_pos_2026-04-07-20-45-53
6,brainbert,sentence_onset,supersubject,DONE,4,2,brainbert_supersubject_sentence_onset_2026-04-...
7,brainbert,whisper_embedding,supersubject,DONE,4,2,brainbert_supersubject_whisper_embedding_2026-...
8,diver,gpt_surprise,supersubject,DONE,4,2,diver_supersubject_gpt_surprise_2026-04-08-07-...
9,diver,content_noncontent,supersubject,DONE,4,2,diver_supersubject_content_noncontent_2026-04-...



All runs complete!


,model,task,variant,status,lags_done,folds_done,missing_lags,latest_run
0,brainbert,gpt_surprise,supersubject,DONE,4,2,[],brainbert_supersubject_gpt_surprise_2026-04-07...
1,brainbert,content_noncontent,supersubject,DONE,4,2,[],brainbert_supersubject_content_noncontent_2026...
2,brainbert,gpt_surprise_multiclass,supersubject,DONE,4,2,[],brainbert_supersubject_gpt_surprise_multiclass...
3,brainbert,iu_boundary,supersubject,DONE,4,2,[],brainbert_supersubject_iu_boundary_2026-04-07-...
4,brainbert,llm_embedding_pretraining,supersubject,DONE,4,2,[],brainbert_supersubject_llm_embedding_pretraini...
5,brainbert,pos,supersubject,DONE,4,2,[],brainbert_supersubject_pos_2026-04-07-20-45-53
6,brainbert,sentence_onset,supersubject,DONE,4,2,[],brainbert_supersubject_sentence_onset_2026-04-...
7,brainbert,whisper_embedding,supersubject,DONE,4,2,[],brainbert_supersubject_whisper_embedding_2026-...
8,diver,gpt_surprise,supersubject,DONE,4,2,[],diver_supersubject_gpt_surprise_2026-04-08-07-...
9,diver,content_noncontent,supersubject,DONE,4,2,[],diver_supersubject_content_noncontent_2026-04-...


In [46]:
import pandas as pd
from pathlib import Path
import re

RESULTS_ROOT = Path("/pscratch/sd/a/ahhyun/EcoGFound/PODCAST/podcast-benchmark/results/foundation_models")

# Task-specific primary metrics
TASK_METRICS = {
    "gpt_surprise":       {"primary": "test_corr_mean",           "secondary": ["test_r2_mean", "test_mse_mean"]},
    "content_noncontent":  {"primary": "test_roc_auc_mean",       "secondary": ["test_acc_logits_mean", "test_f1_logits_mean"]},
    "llm_decoding":        {"primary": "test_top_10_acc_llm_mean","secondary": ["test_cross_entropy_mean"]},
    "sentence_onset":      {"primary": "test_corr_mean",          "secondary": ["test_r2_mean", "test_mse_mean"]},
    "volume_level":        {"primary": "test_corr_mean",          "secondary": ["test_r2_mean", "test_mse_mean"]}
}

rows = []
for csv_path in sorted(RESULTS_ROOT.rglob("lag_performance.csv")):
    rel = csv_path.relative_to(RESULTS_ROOT)
    parts = rel.parts
    model, task, variant = parts[0], parts[1], parts[2]
    run_dir = parts[3]
    ts_match = re.search(r'(\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2})$', run_dir)
    timestamp = ts_match.group(1) if ts_match else run_dir

    df = pd.read_csv(csv_path)
    metrics_info = TASK_METRICS.get(task, {})
    primary_col = metrics_info.get("primary", "test_corr_mean")
    secondary_cols = metrics_info.get("secondary", [])

    for _, row in df.iterrows():
        entry = {
            "model": model,
            "task": task,
            "variant": variant,
            "timestamp": timestamp,
            "lag": row["lags"],
            "primary_metric_name": primary_col.replace("test_", "").replace("_mean", ""),
            "primary_metric": row.get(primary_col),
            "epochs": row.get("num_epochs_mean"),
        }
        for col in secondary_cols:
            entry[col.replace("test_", "").replace("_mean", "")] = row.get(col)
        rows.append(entry)

nersc = pd.DataFrame(rows)
nersc["source"] = "nersc"

# Filter: only runs after 1:30 AM EDT Apr 7 2026 (= 10:30 PM PDT Apr 6)
nersc = nersc[nersc["timestamp"] >= "2026-04-06-22-30-00"]
if nersc.empty:
    print("NERSC: No results found — check RESULTS_ROOT path")
else:                                                                                                                                                                    
    print(f"NERSC: Loaded {len(nersc)} rows from {nersc[['model','task','variant','timestamp']].drop_duplicates().shape[0]} runs")
print(f"NERSC: Loaded {len(nersc)} rows from {nersc[['model','task','variant','timestamp']].drop_duplicates().shape[0]} runs")
nersc

NERSC: Loaded 118 rows from 85 runs
NERSC: Loaded 118 rows from 85 runs


,model,task,variant,timestamp,lag,primary_metric_name,primary_metric,epochs,acc_logits,f1_logits,r2,mse,cross_entropy,source
2,brainbert,content_noncontent,supersubject,2026-04-06-23-34-08,-250.0,roc_auc,0.524802,3.0,0.457837,0.323280,NaN,NaN,NaN,nersc
3,brainbert,content_noncontent,supersubject,2026-04-06-23-34-08,0.0,roc_auc,0.506118,7.0,0.503968,0.482691,NaN,NaN,NaN,nersc
4,brainbert,content_noncontent,supersubject,2026-04-06-23-34-08,250.0,roc_auc,0.495536,3.5,0.466270,0.397354,NaN,NaN,NaN,nersc
5,brainbert,content_noncontent,supersubject,2026-04-06-23-34-08,500.0,roc_auc,0.524967,17.0,0.530258,0.463539,NaN,NaN,NaN,nersc
6,brainbert,content_noncontent,supersubject,2026-04-07-09-01-15,500.0,roc_auc,0.507275,6.5,0.546627,0.421447,NaN,NaN,NaN,nersc
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,popt,sentence_onset,supersubject,2026-04-07-09-18-48,500.0,corr,NaN,2.0,NaN,NaN,NaN,NaN,NaN,nersc
137,popt,sentence_onset,supersubject,2026-04-07-11-21-38,250.0,corr,NaN,2.0,NaN,NaN,NaN,NaN,NaN,nersc
138,popt,sentence_onset,supersubject,2026-04-07-11-22-10,0.0,corr,NaN,1.5,NaN,NaN,NaN,NaN,NaN,nersc
139,popt,whisper_embedding,supersubject,2026-04-07-09-23-54,500.0,corr,0.046414,88.0,NaN,NaN,NaN,NaN,NaN,nersc


In [47]:
# Compare models on common lag×fold combinations for a given task
# Only includes lag×fold pairs where ALL models have results

FILTER_TASK = "gpt_surprise"  # <-- change this
COMPARE_MODELS = ["brainbert", "diver", "popt"]

# Use latest run per (model, task, variant, lag)
latest_nersc = nersc.sort_values("timestamp").groupby(["model", "task", "variant", "lag"]).last().reset_index()
task_df = latest_nersc[(latest_nersc["task"] == FILTER_TASK) & (latest_nersc["model"].isin(COMPARE_MODELS))].copy()

if task_df.empty:
    print(f"No results for task '{FILTER_TASK}'")
else:
    metric_name = task_df["primary_metric_name"].iloc[0]

    # Find lags that exist for ALL models
    lags_per_model = task_df.groupby("model")["lag"].apply(set)
    common_lags = sorted(set.intersection(*lags_per_model))
    all_lags = sorted(set.union(*lags_per_model))
    excluded_lags = sorted(set(all_lags) - set(common_lags))

    print(f"Task: {FILTER_TASK} | Metric: {metric_name}")
    print(f"Common lags (present in all models): {common_lags}")
    if excluded_lags:
        print(f"Excluded lags (not in all models): {excluded_lags}")
        for lag in excluded_lags:
            have = sorted(task_df[task_df["lag"] == lag]["model"].unique())
            missing = sorted(set(COMPARE_MODELS) - set(have))
            print(f"  lag {lag}: missing {missing}")
    print(f"Folds: [1, 5]")
    print()

    # Filter to common lags only
    common_df = task_df[task_df["lag"].isin(common_lags)]

    # Per-model average across common lags
    summary = common_df.groupby("model").agg(
        mean_metric=("primary_metric", "mean"),
        std_metric=("primary_metric", "std"),
        n_lags=("lag", "nunique"),
        lags=("lag", lambda x: sorted(x.unique())),
    ).reset_index()
    summary.columns = ["model", f"mean_{metric_name}", f"std_{metric_name}", "n_lags", "lags"]
    print("=== Average across common lags ===")
    display(summary)

    # Pivot: model × lag
    print(f"\n=== {metric_name} per lag ===")
    pivot = common_df.pivot_table(index="model", columns="lag", values="primary_metric", aggfunc="first")
    display(pivot.style.format("{:.4f}", na_rep="-").background_gradient(cmap="RdYlGn", axis=None))


Task: gpt_surprise | Metric: corr
Common lags (present in all models): [0.0, 250.0]
Excluded lags (not in all models): [-250.0, 500.0]
  lag -250.0: missing ['popt']
  lag 500.0: missing ['diver']
Folds: [1, 5]

=== Average across common lags ===


,model,mean_corr,std_corr,n_lags,lags
0,brainbert,0.001669,0.018665,2,"[0.0, 250.0]"
1,diver,0.043545,0.010945,2,"[0.0, 250.0]"
2,popt,0.006029,0.009384,2,"[0.0, 250.0]"



=== corr per lag ===


lag,0.000000,250.000000
model,,
brainbert,0.0149,-0.0115
diver,0.0513,0.0358
popt,-0.0006,0.0127


In [48]:
nersc[(nersc['task'] == 'gpt_surprise') & (nersc['variant']=="supersubject")].sort_values('primary_metric', ascending=False)

,model,task,variant,timestamp,lag,primary_metric_name,primary_metric,epochs,acc_logits,f1_logits,r2,mse,cross_entropy,source
64,diver,gpt_surprise,supersubject,2026-04-07-00-12-42,-250.0,corr,0.056063,11.0,NaN,NaN,-4.289692,0.035590,NaN,nersc
65,diver,gpt_surprise,supersubject,2026-04-07-10-53-38,0.0,corr,0.051284,10.0,NaN,NaN,-4.178925,0.035834,NaN,nersc
66,diver,gpt_surprise,supersubject,2026-04-07-11-01-22,250.0,corr,0.035806,14.0,NaN,NaN,-4.278580,0.035978,NaN,nersc
18,brainbert,gpt_surprise,supersubject,2026-04-06-23-47-54,0.0,corr,0.028922,1.0,NaN,NaN,-22.635035,0.280906,NaN,nersc
15,brainbert,gpt_surprise,supersubject,2026-04-06-23-33-24,0.0,corr,0.022689,4.0,NaN,NaN,-8.285984,0.080033,NaN,nersc
20,brainbert,gpt_surprise,supersubject,2026-04-07-10-55-21,0.0,corr,0.014867,6.0,NaN,NaN,-10.423592,0.078381,NaN,nersc
21,brainbert,gpt_surprise,supersubject,2026-04-07-10-56-49,0.0,corr,0.014867,6.0,NaN,NaN,-10.423592,0.078381,NaN,nersc
102,popt,gpt_surprise,supersubject,2026-04-07-11-03-30,250.0,corr,0.012664,67.0,NaN,NaN,-4.245564,0.036342,NaN,nersc
14,brainbert,gpt_surprise,supersubject,2026-04-06-23-33-24,-250.0,corr,0.012217,4.5,NaN,NaN,-10.215325,0.074348,NaN,nersc
16,brainbert,gpt_surprise,supersubject,2026-04-06-23-33-24,250.0,corr,0.011110,3.0,NaN,NaN,-11.584459,0.071796,NaN,nersc


In [49]:
# Load labserver results
labserver = pd.read_csv("podcast_benchmark_from_labserver.csv")
labserver["source"] = "labserver"
print(f"Labserver: {len(labserver)} rows from {labserver[['model','task','variant','timestamp']].drop_duplicates().shape[0]} runs")

# Combine
results = pd.concat([nersc, labserver], ignore_index=True)
print(f"\nCombined: {len(results)} total rows")

Labserver: 101 rows from 101 runs

Combined: 219 total rows


In [50]:
#labserver[labserver.task=='content_noncontent']

In [51]:
#results[(results['lag'] == 0) & (results['task']=='content_noncontent')] #& (results['epochs'] == 1.0)

In [52]:
# Latest run per (source, model, task, variant)
latest = results.sort_values("timestamp").groupby(["source", "model", "task", "variant", "lag"]).last().reset_index()
display_cols = ["source", "model", "task", "variant", "lag", "primary_metric_name", "primary_metric", "epochs"]
extra = [c for c in latest.columns if c not in display_cols and c not in ["timestamp"]]
latest[display_cols + extra]

,source,model,task,variant,lag,primary_metric_name,primary_metric,epochs,acc_logits,f1_logits,r2,mse,cross_entropy
0,labserver,brainbert,content_noncontent,subject1_full,0.0,roc_auc,0.458003,19.0,0.464286,0.401852,NaN,NaN,NaN
1,labserver,brainbert,content_noncontent,supersubject,0.0,roc_auc,0.481151,5.0,0.484127,0.463889,NaN,NaN,NaN
2,labserver,brainbert,gpt_surprise,supersubject,0.0,corr,0.034398,4.0,NaN,NaN,-6.528771,0.069302,NaN
3,labserver,brainbert,gpt_surprise_multiclass,supersubject,0.0,corr,NaN,11.0,NaN,NaN,NaN,NaN,NaN
4,labserver,brainbert,iu_boundary,supersubject,0.0,corr,NaN,11.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
104,nersc,popt,sentence_onset,supersubject,0.0,corr,NaN,1.5,NaN,NaN,NaN,NaN,NaN
105,nersc,popt,sentence_onset,supersubject,250.0,corr,NaN,2.0,NaN,NaN,NaN,NaN,NaN
106,nersc,popt,sentence_onset,supersubject,500.0,corr,NaN,2.0,NaN,NaN,NaN,NaN,NaN
107,nersc,popt,whisper_embedding,supersubject,0.0,corr,0.047506,76.5,NaN,NaN,NaN,NaN,NaN


In [53]:
# Side-by-side comparison: NERSC vs Labserver (latest runs, lag=0)
lag0 = latest[latest["lag"] == 0].copy()

comparison = lag0.pivot_table(
    index=["model", "task", "variant"],
    columns="source",
    values="primary_metric",
    aggfunc="first"
).reset_index()

if "nersc" in comparison.columns and "labserver" in comparison.columns:
    comparison["diff"] = comparison["nersc"] - comparison["labserver"]
    comparison["match"] = comparison["diff"].abs() < 0.01

# Add metric name for context
metric_map = lag0.drop_duplicates("task").set_index("task")["primary_metric_name"]
comparison["metric"] = comparison["task"].map(metric_map)

print("NERSC vs Labserver comparison (lag=0, latest runs)")
print("match = |diff| < 0.01\n")
comparison

NERSC vs Labserver comparison (lag=0, latest runs)
match = |diff| < 0.01



source,model,task,variant,labserver,nersc,diff,match,metric
0,brainbert,content_noncontent,subject1_full,0.458003,NaN,NaN,False,roc_auc
1,brainbert,content_noncontent,supersubject,0.481151,0.481978,0.000827,True,roc_auc
2,brainbert,gpt_surprise,supersubject,0.034398,0.014867,-0.019531,False,corr
3,brainbert,llm_decoding,supersubject,0.570201,NaN,NaN,False,top_10_acc_llm
4,brainbert,llm_embedding_pretraining,supersubject,0.416470,0.164225,-0.252245,False,corr
5,brainbert,volume_level,supersubject,0.180498,NaN,NaN,False,corr
6,brainbert,whisper_embedding,supersubject,0.061037,0.065642,0.004605,True,corr
7,brainbert,word_embedding,supersubject,0.004895,NaN,NaN,False,corr
8,diver,content_noncontent,subject1_full,0.466931,NaN,NaN,False,roc_auc
9,diver,content_noncontent,supersubject,0.522817,NaN,NaN,False,roc_auc


In [54]:
# Pivot heatmap: primary_metric by model x task, per source
for source in ["nersc", "labserver"]:
    src = lag0[lag0["source"] == source]
    if src.empty:
        continue
    pivot = src.pivot_table(index="model", columns="task", values="primary_metric", aggfunc="first")
    metric_names = src.drop_duplicates("task").set_index("task")["primary_metric_name"]
    print(f"\n=== {source.upper()} ===")
    print("Metrics:", dict(metric_names))
    display(pivot.style.format("{:.4f}", na_rep="-").background_gradient(cmap="RdYlGn", axis=None))


=== NERSC ===
Metrics: {'content_noncontent': 'roc_auc', 'gpt_surprise': 'corr', 'gpt_surprise_multiclass': 'corr', 'iu_boundary': 'corr', 'llm_embedding_pretraining': 'corr', 'sentence_onset': 'corr', 'whisper_embedding': 'corr', 'pos': 'corr'}


task,content_noncontent,gpt_surprise,llm_embedding_pretraining,whisper_embedding
model,,,,
brainbert,0.4820,0.0149,0.1642,0.0656
diver,-,0.0513,-,-
popt,0.5106,-0.0006,0.5487,0.0475



=== LABSERVER ===
Metrics: {'content_noncontent': 'roc_auc', 'gpt_surprise': 'corr', 'gpt_surprise_multiclass': 'corr', 'iu_boundary': 'corr', 'llm_decoding': 'top_10_acc_llm', 'llm_embedding_pretraining': 'corr', 'pos': 'corr', 'sentence_onset': 'corr', 'volume_level': 'corr', 'whisper_embedding': 'corr', 'word_embedding': 'corr'}


task,content_noncontent,gpt_surprise,llm_decoding,llm_embedding_pretraining,volume_level,whisper_embedding,word_embedding
model,,,,,,,
brainbert,0.4580,0.0344,0.5702,0.4165,0.1805,0.0610,0.0049
diver,0.4669,0.0020,0.5769,0.5591,0.1365,0.1124,0.0775
popt,0.4993,-0.0484,0.5624,0.5695,-,-0.0005,-0.0839


In [55]:
only_early_stopped_not_debug_df = nersc[nersc["epochs"] > 10] 
df = only_early_stopped_not_debug_df
df

,model,task,variant,timestamp,lag,primary_metric_name,primary_metric,epochs,acc_logits,f1_logits,r2,mse,cross_entropy,source
5,brainbert,content_noncontent,supersubject,2026-04-06-23-34-08,500.0,roc_auc,0.524967,17.0,0.530258,0.463539,NaN,NaN,NaN,nersc
24,brainbert,gpt_surprise_multiclass,supersubject,2026-04-06-23-33-55,0.0,corr,NaN,12.5,NaN,NaN,NaN,NaN,NaN,nersc
25,brainbert,gpt_surprise_multiclass,supersubject,2026-04-07-09-01-18,500.0,corr,NaN,18.0,NaN,NaN,NaN,NaN,NaN,nersc
27,brainbert,gpt_surprise_multiclass,supersubject,2026-04-07-11-04-48,250.0,corr,NaN,13.5,NaN,NaN,NaN,NaN,NaN,nersc
28,brainbert,iu_boundary,supersubject,2026-04-06-23-33-49,-250.0,corr,NaN,17.0,NaN,NaN,NaN,NaN,NaN,nersc
29,brainbert,iu_boundary,supersubject,2026-04-06-23-33-49,0.0,corr,NaN,11.5,NaN,NaN,NaN,NaN,NaN,nersc
30,brainbert,iu_boundary,supersubject,2026-04-06-23-33-49,250.0,corr,NaN,16.0,NaN,NaN,NaN,NaN,NaN,nersc
33,brainbert,iu_boundary,supersubject,2026-04-07-11-05-50,0.0,corr,NaN,15.0,NaN,NaN,NaN,NaN,NaN,nersc
37,brainbert,llm_embedding_pretraining,supersubject,2026-04-06-23-33-28,-250.0,corr,0.159468,13.0,NaN,NaN,NaN,NaN,NaN,nersc
38,brainbert,llm_embedding_pretraining,supersubject,2026-04-07-09-17-53,500.0,corr,0.180086,15.0,NaN,NaN,NaN,NaN,NaN,nersc


In [56]:
# Filter by task -> average primary_metric across models, per lag
# Uses latest run per (model, task, variant, lag)

FILTER_TASK = "gpt_surprise"  # <-- change this

latest_nersc = nersc.sort_values("timestamp").groupby(["model", "task", "variant", "lag"]).last().reset_index()
task_df = latest_nersc[latest_nersc["task"] == FILTER_TASK].copy()

if task_df.empty:
    print(f"No results for task '{FILTER_TASK}'")
else:
    models_included = sorted(task_df["model"].unique())
    lags_included = sorted(task_df["lag"].unique())
    variants_included = sorted(task_df["variant"].unique())
    metric_name = task_df["primary_metric_name"].iloc[0]

    print(f"Task: {FILTER_TASK}")
    print(f"Metric: {metric_name}")
    print(f"Models averaged: {models_included}")
    print(f"Variants: {variants_included}")
    print(f"Lags: {lags_included}")
    print(f"Folds: [1, 5]")  # from --fold-ids [1,5]
    print()

    # Average across models per lag
    avg = task_df.groupby("lag").agg(
        primary_metric_mean=("primary_metric", "mean"),
        primary_metric_std=("primary_metric", "std"),
        n_models=("model", "nunique"),
        models=("model", lambda x: ", ".join(sorted(x.unique()))),
    ).reset_index()
    display(avg)

    # Per-model breakdown
    print("\nPer-model breakdown:")
    breakdown = task_df[["model", "lag", "primary_metric", "epochs", "timestamp"]].sort_values(["lag", "model"])
    display(breakdown)


Task: gpt_surprise
Metric: corr
Models averaged: ['brainbert', 'diver', 'popt']
Variants: ['supersubject']
Lags: [-250.0, 0.0, 250.0, 500.0]
Folds: [1, 5]



,lag,primary_metric_mean,primary_metric_std,n_models,models
0,-250.0,0.034140,0.031004,2,"brainbert, diver"
1,0.0,0.021848,0.026640,3,"brainbert, diver, popt"
2,250.0,0.012313,0.023670,3,"brainbert, diver, popt"
3,500.0,-0.000584,0.001728,2,"brainbert, popt"



Per-model breakdown:


,model,lag,primary_metric,epochs,timestamp
4,brainbert,-250.0,0.012217,4.5,2026-04-06-23-33-24
32,diver,-250.0,0.056063,11.0,2026-04-07-00-12-42
5,brainbert,0.0,0.014867,6.0,2026-04-07-10-56-49
33,diver,0.0,0.051284,10.0,2026-04-07-10-53-38
50,popt,0.0,-0.000607,68.5,2026-04-07-10-57-49
6,brainbert,250.0,-0.011529,2.0,2026-04-07-11-04-47
34,diver,250.0,0.035806,14.0,2026-04-07-11-01-22
51,popt,250.0,0.012664,67.0,2026-04-07-11-03-30
7,brainbert,500.0,0.000638,7.0,2026-04-07-09-01-03
52,popt,500.0,-0.001806,37.0,2026-04-07-11-11-09
